# Step 1 - Grant schema-level access

In [0]:
%sql
-- Engineers: full access to silver
GRANT USE CATALOG ON CATALOG ecommerce TO `ecommerce_engineers`;
GRANT USE SCHEMA ON SCHEMA ecommerce.silver TO `ecommerce_engineers`;
GRANT ALL PRIVILEGES ON SCHEMA ecommerce.silver TO `ecommerce_engineers`;

-- Analysts: read-only access to gold
GRANT USE CATALOG ON CATALOG ecommerce TO `ecommerce_analysts`;
GRANT USE SCHEMA ON SCHEMA ecommerce.gold TO `ecommerce_analysts`;
GRANT SELECT ON SCHEMA ecommerce.gold TO `ecommerce_analysts`;

# Step 2 - Column masking on customer ZIP codes

In [0]:
%sql
CREATE OR REPLACE FUNCTION ecommerce.silver.mask_zip_code(zip STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('ecommerce_engineers') THEN zip
    ELSE CONCAT(LEFT(zip, 2), '***')
END;

-- Apply the mask to the customers table
ALTER TABLE ecommerce.silver.customers
ALTER COLUMN customer_zip_code_prefix
SET MASK ecommerce.silver.mask_zip_code;

DESCRIBE EXTENDED ecommerce.silver.customers;

In [0]:
# Run as member of 'ecommerce_engineers', remove yourself from group, then run this cell again to verify columns are masked correctly, then add yourself back to group
display(spark.read.table(f"ecommerce.silver.customers").show(10))

In [0]:
%sql
SELECT * FROM ecommerce.silver.customers LIMIT 5;

# Step 3 - Row-level security on seller performance

In [0]:
%sql
CREATE OR REPLACE FUNCTION ecommerce.gold.seller_row_filter(seller_id STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('ecommerce_engineers')
    -- OR is_account_group_member('admins')
    OR current_user() = seller_id;

-- Attach filter
ALTER TABLE ecommerce.gold.seller_performance
SET ROW FILTER ecommerce.gold.seller_row_filter ON (seller_id);

In [0]:
%sql
SELECT * FROM ecommerce.gold.seller_performance LIMIT 10;

# Checkpoints

In [0]:
%sql
SELECT * FROM ecommerce.silver.customers LIMIT 5; -- → should show masked ZIP codes
SELECT * FROM ecommerce.gold.seller_performance; -- → should return zero rows
SELECT * FROM ecommerce.gold.monthly_revenue LIMIT 5; -- → should succeed (analyst has SELECT on gold)

# Challenge: ABAC Tag-Based Policies

1. Create a governed tag (pii:zip_code)
2. Tag the sensitive columns
    ```python
    %sql
    ALTER TABLE ecommerce.silver.customers
    ALTER COLUMN customer_zip_code_prefix
    SET TAGS ('pii' = 'zip_code');  
    ```
3. Create UDF (user defined function) for masking the zip codes
    ```
    CREATE OR REPLACE FUNCTION ecommerce.silver.mask_zip_code(zip STRING)
    RETURNS STRING
    RETURN CASE
        WHEN is_account_group_member('ecommerce_engineers') THEN zip
        ELSE CONCAT(LEFT(zip, 2), '***')
    END;
    ```
4. Create the Mask Policy
    1. Go to Policies → New policy
    2. Name: mask_zip_code
    3. Applied to: All account users
    4. Scope: Catalog = test, schema = default, All tables
    5. Purpose: Mask column data
    6. Condition: Mask column if tagged with pii:zip_code
    7. Function: Select existing = mask_zip_code

> Link: https://medium.com/@cralle/a-step-by-step-guide-to-setting-up-abac-in-databricks-unity-catalog-8266454bc87f$0

In [0]:
%sql
ALTER TABLE ecommerce.silver.customers
ALTER COLUMN customer_zip_code_prefix
SET TAGS ('pii' = 'zip_code');

In [0]:
%sql
SELECT * FROM ecommerce.silver.customers LIMIT 5;